In [ ]:
import pandas as pd
import time
import requests
from io import StringIO

In [ ]:
def get_mvp_table(year):
    url = f"https://www.basketball-reference.com/awards/awards_{year}.html"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.basketball-reference.com/awards/"
    }
    
    response = requests.get(url, headers=headers)
    
    print(year, response.status_code)
    
    tables = pd.read_html(StringIO(response.text)) #asks pandas to find all tables inside webpage
    #StringIO makes it behave like a readable file for pandas
    mvp_table = tables[0]
    
    if isinstance(mvp_table.columns, pd.MultiIndex):
        mvp_table.columns = mvp_table.columns.droplevel(0)
    
    mvp_table["Season"] = year
    
    return mvp_table

## Scraping MVP Voting Data

The first dataset comes from Basketball Reference's MVP voting tables. Each seasons awards page contains the players who received MVP votes, along with their vote share and season statistics.

In [ ]:
mvp_2025 = get_mvp_table(2025)
mvp_2025.head(10)

### Building the MVP Voting Dataset

The following loop collects MVP voting tables from 2010 through 2026 and combines them in to a single dataframe. Each row represents a player who received MVP votes in a given season. NOTE that Basketball Reference is sometimes blocked so I had to add a time.sleep, hence running this cell takes a minute (by design)

In [ ]:
years = range(2010, 2027)

all_mvp_tables = []

for year in years:
    table = get_mvp_table(year)
    all_mvp_tables.append(table)
    time.sleep(6)

all_mvp = pd.concat(all_mvp_tables, ignore_index=True)

all_mvp.head()

In [ ]:
all_mvp.to_csv("all_mvp_2010_2026.csv", index=False) #to save incase blocked

## Selecting Model Variables

From the full MVP voting table I pselected player statistics, efficiency metrics, win-share metrics, vote share, and season. The target variable in the regression (the Y) is MVP vote share.

In [ ]:
model_cols = [
    "Player", "Age", "Tm", "G", "MP",
    "PTS", "TRB", "AST", "STL", "BLK",
    "FG%", "3P%", "FT%", "WS", "WS/48",
    "Share", "Season"
]

mvp_model_df = all_mvp[model_cols].copy()

mvp_model_df.head()

In [ ]:
#checks data types bc sometimes scraped tables have "numbers" stored as text
mvp_model_df.dtypes

In [ ]:
#corrects missing values and confirms cleanliness
mvp_model_df = mvp_model_df.dropna().copy()
mvp_model_df.isna().sum()

## Model 1: First Statistical Regression (Broad)

The first model uses a larger set of player statistics including scoring, efficiency, availability, and winshare metrics. This version gives the model a broad statistical view of each MVP candidate, but many of these variables overlap with each other which makes practical results annoying.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

features = [
    "Age", "G", "MP",
    "PTS", "TRB", "AST", "STL", "BLK",
    "FG%", "3P%", "FT%", "WS", "WS/48"
]

X = mvp_model_df[features]
y = mvp_model_df["Share"]

model = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

model.fit(X, y); #this is where python solves the least squares problem

#so our model is taking stat columns in X, standardizing them, feeding them into linear regression, learning
#coefficients to predict share

In [ ]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.named_steps["linearregression"].coef_
})

coef_df["Abs Coefficient"] = coef_df["Coefficient"].abs()

coef_df.sort_values("Abs Coefficient", ascending=False)

Because several features measure very interlinked concepts (such as scoring volume, efficiency, win shares, etc) so the coefficients should not be interpreted as pure isolated effects. Later models in the notebook will clean this up.

In [ ]:
mvp_model_df["Predicted Share"] = model.predict(X) 
#^uses trained model to predict mvp vote share for each player

mvp_model_df["Prediction Error"] = mvp_model_df["Share"] - mvp_model_df["Predicted Share"]

mvp_model_df[mvp_model_df["Season"] == 2026][[
    "Player", "Season", "Share", "Predicted Share", "Prediction Error"
]].sort_values("Predicted Share", ascending=False)

#Generate Model 1 predictions for each MVP vote getter

In [ ]:
mvp_model_df[mvp_model_df["Season"] == 2025][[
    "Player", "Share", "Predicted Share", "Prediction Error"
]].sort_values("Predicted Share", ascending=False)

#how does it predict 2025?

We can see issues here for example. Model 1 slightly favored Jokic over Shai in 2025 even though the actual vote heavily favored Shai. This suggests that the broad stat only model may overweight all around statistical profiles.

(also note I began this a day or two before 2026 MVP results released but '25 and '26 do show similar Jokic-biased errors which makes some sense)

Next step is to train on previous seasons and test on a held out season instead

In [ ]:
train_df = mvp_model_df[mvp_model_df["Season"] < 2026].copy()
test_df = mvp_model_df[mvp_model_df["Season"] == 2026].copy()

X_train = train_df[features]
y_train = train_df["Share"]

X_test = test_df[features]
y_test = test_df["Share"]

model_v1_test = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

model_v1_test.fit(X_train, y_train);

test_df["Predicted Share"] = model_v1_test.predict(X_test)
test_df["Prediction Error"] = test_df["Share"] - test_df["Predicted Share"]

test_df[[
    "Player", "Share", "Predicted Share", "Prediction Error"
]].sort_values("Predicted Share", ascending=False)

## Model 2: Cleaner Production-Based Feature Set

Model 2 uses a smaller set of core player production variables. The goal is to reduce overlap between similar statistics while still capturing scoring, playmaking, defense, rebounding, and overall impact.

In [ ]:
features_v2 = [
    "PTS", "TRB", "AST", "STL", "BLK",
    "WS"
]

X_train_v2 = train_df[features_v2]
y_train_v2 = train_df["Share"]

X_test_v2 = test_df[features_v2]
y_test_v2 = test_df["Share"]

model_v2_test = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

model_v2_test.fit(X_train_v2, y_train_v2)

test_df["Predicted Share V2"] = model_v2_test.predict(X_test_v2)
test_df["Prediction Error V2"] = test_df["Share"] - test_df["Predicted Share V2"]

test_df[[
    "Player", "Share", "Predicted Share V2", "Prediction Error V2"
]].sort_values("Predicted Share V2", ascending=False)

In [ ]:
from sklearn.metrics import mean_absolute_error

def backtest_model(feature_list, start_year=2015, end_year=2026):
    results = []

    for season in range(start_year, end_year + 1):
        train = mvp_model_df[mvp_model_df["Season"] < season].copy()
        test = mvp_model_df[mvp_model_df["Season"] == season].copy()

        X_train = train[feature_list]
        y_train = train["Share"]
        X_test = test[feature_list]

        model = make_pipeline(
            StandardScaler(),
            LinearRegression()
        )

        model.fit(X_train, y_train)

        test["Predicted Share"] = model.predict(X_test)

        actual_winner = test.sort_values("Share", ascending=False).iloc[0]
        predicted_winner = test.sort_values("Predicted Share", ascending=False).iloc[0]

        results.append({
            "Season": season,
            "Actual MVP": actual_winner["Player"],
            "Predicted MVP": predicted_winner["Player"],
            "Correct": actual_winner["Player"] == predicted_winner["Player"],
            "MAE": mean_absolute_error(test["Share"], test["Predicted Share"])
        })

    return pd.DataFrame(results)

In [ ]:
v2_results = backtest_model(features_v2)

v2_results

In [ ]:
print("Model 2 Results")
print("------------------------")
print("Correct rate:", round(v2_results["Correct"].mean(), 3))
print("Average MAE:", round(v2_results["MAE"].mean(), 3))

## Model 3: Adding Team Success

Model 3 adds team winning percentage as a predictor. MVP voting often reflects both individual production and team success, which checks out. NOTE once again the neccessary time.sleep makes this cell take a minute to run so Basketball Reference doesn't throw issues

In [ ]:
team_name_to_abbrev = {
    "Atlanta Hawks": "ATL",
    "Boston Celtics": "BOS",
    "Brooklyn Nets": "BRK",
    "New Jersey Nets": "NJN",
    "Charlotte Bobcats": "CHA",
    "Charlotte Hornets": "CHO",
    "Chicago Bulls": "CHI",
    "Cleveland Cavaliers": "CLE",
    "Dallas Mavericks": "DAL",
    "Denver Nuggets": "DEN",
    "Detroit Pistons": "DET",
    "Golden State Warriors": "GSW",
    "Houston Rockets": "HOU",
    "Indiana Pacers": "IND",
    "Los Angeles Clippers": "LAC",
    "Los Angeles Lakers": "LAL",
    "Memphis Grizzlies": "MEM",
    "Miami Heat": "MIA",
    "Milwaukee Bucks": "MIL",
    "Minnesota Timberwolves": "MIN",
    "New Orleans Hornets": "NOH",
    "New Orleans Pelicans": "NOP",
    "New York Knicks": "NYK",
    "Oklahoma City Thunder": "OKC",
    "Orlando Magic": "ORL",
    "Philadelphia 76ers": "PHI",
    "Phoenix Suns": "PHO",
    "Portland Trail Blazers": "POR",
    "Sacramento Kings": "SAC",
    "San Antonio Spurs": "SAS",
    "Toronto Raptors": "TOR",
    "Utah Jazz": "UTA",
    "Washington Wizards": "WAS"
}

def get_team_records(year):
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_standings.html"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.basketball-reference.com/"
    }
    
    response = requests.get(url, headers=headers)
    print(year, response.status_code)
    
    tables = pd.read_html(StringIO(response.text))
    
    east = tables[0]
    west = tables[1]
    
    east = east.rename(columns={east.columns[0]: "Team"})
    west = west.rename(columns={west.columns[0]: "Team"})
    
    records = pd.concat([east, west], ignore_index=True)
    
    records["Team"] = (
        records["Team"]
        .astype(str)
        .str.replace("\xa0", " ", regex=False)
        .str.replace("*", "", regex=False)
        .str.replace(r"\s*\(\d+\)", "", regex=True)
        .str.strip()
    )
    
    records["Tm"] = records["Team"].map(team_name_to_abbrev)
    records["Season"] = year
    
    records = records.dropna(subset=["Tm"]).copy()
    
    records = records[["Season", "Tm", "W", "L", "W/L%"]].copy()
    
    records["W"] = pd.to_numeric(records["W"], errors="coerce")
    records["L"] = pd.to_numeric(records["L"], errors="coerce")
    records["W/L%"] = pd.to_numeric(records["W/L%"], errors="coerce")
    
    return records


all_team_records = []

for year in range(2010, 2027):
    records = get_team_records(year)
    all_team_records.append(records)
    time.sleep(6)

team_records = pd.concat(all_team_records, ignore_index=True)

In [ ]:
#merging mvp_model_df with team_records and matching rows where season and team abbrev match
mvp_model_df_v3 = mvp_model_df[model_cols].merge(
    team_records[["Season", "Tm", "W/L%"]],
    on=["Season", "Tm"],
    how="left"
)

mvp_model_df_v3 = mvp_model_df_v3.dropna(subset=["W/L%"]).copy()

mvp_model_df_v3.head()

In [ ]:
# quick name cleanup for display
def clean_player_names(df):
    df["Player"] = df["Player"].astype(str)
    df["Player"] = df["Player"].str.replace(r"Nikola Joki.*", "Nikola Jokic", regex=True)
    df["Player"] = df["Player"].str.replace(r"Luka Don.*", "Luka Doncic", regex=True)
    df["Player"] = df["Player"].str.replace(r"Manu Gin.*", "Manu Ginobili", regex=True)
    return df

mvp_model_df = clean_player_names(mvp_model_df)
mvp_model_df_v3 = clean_player_names(mvp_model_df_v3)

## Final Model 3 Backtest

The final version of the model uses core player production statistics plus team winning percentage. It's backtested from 2015 through 2026 by training only on seasons before the year being tested.

In [ ]:
def backtest_model_v3(feature_list, start_year=2015, end_year=2026):
    results = []

    for season in range(start_year, end_year + 1):
        train = mvp_model_df_v3[mvp_model_df_v3["Season"] < season].copy()
        test = mvp_model_df_v3[mvp_model_df_v3["Season"] == season].copy()

        X_train = train[feature_list]
        y_train = train["Share"]
        X_test = test[feature_list]

        model = make_pipeline(
            StandardScaler(),
            LinearRegression()
        )

        model.fit(X_train, y_train)

        test["Predicted Share"] = model.predict(X_test)

        actual_winner = test.sort_values("Share", ascending=False).iloc[0]
        predicted_winner = test.sort_values("Predicted Share", ascending=False).iloc[0]

        results.append({
            "Season": season,
            "Actual MVP": actual_winner["Player"],
            "Predicted MVP": predicted_winner["Player"],
            "Correct": actual_winner["Player"] == predicted_winner["Player"],
            "MAE": mean_absolute_error(test["Share"], test["Predicted Share"])
        })

    return pd.DataFrame(results)

features_v3 = [
    "PTS", "TRB", "AST", "STL", "BLK",
    "WS", "W/L%"
]

v3_results = backtest_model_v3(features_v3, start_year=2015, end_year=2026)

model3_correct = v3_results["Correct"].mean()
model3_avg_mae = v3_results["MAE"].mean()
model3_correct_count = v3_results["Correct"].sum()
model3_total = len(v3_results)

print("Model 3 Results")
print("----------------")
print("Correct MVP picks:", model3_correct_count, "out of", model3_total)
print("Correct rate:", round(model3_correct, 3))
print("Average MAE:", round(model3_avg_mae, 3))

display(v3_results)

# final 2026 prediction table: train through 2025, test on 2026
train_v3_final = mvp_model_df_v3[mvp_model_df_v3["Season"] < 2026].copy()
test_2026 = mvp_model_df_v3[mvp_model_df_v3["Season"] == 2026].copy()

X_train_v3_final = train_v3_final[features_v3]
y_train_v3_final = train_v3_final["Share"]

model_v3_final = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

model_v3_final.fit(X_train_v3_final, y_train_v3_final)

test_2026["Predicted Share"] = model_v3_final.predict(test_2026[features_v3])
test_2026["Prediction Error"] = test_2026["Share"] - test_2026["Predicted Share"]

display(
    test_2026[
        ["Player", "Tm", "Share", "Predicted Share", "Prediction Error"]
    ].sort_values("Predicted Share", ascending=False)
)

coef_v3_df = pd.DataFrame({
    "Feature": features_v3,
    "Coefficient": model_v3_final.named_steps["linearregression"].coef_
})

coef_v3_df["Abs Coefficient"] = coef_v3_df["Coefficient"].abs()

display(coef_v3_df.sort_values("Abs Coefficient", ascending=False))